# Datathon 2026 – Task 2: Wikipedia Next Article Prediction
**Task**: Prediksi artikel Wikipedia mana yang akan diklik user selanjutnya, berdasarkan artikel saat ini (`current_article_id`) dan tujuan (`target_article_id`).

**Pendekatan**: Transition frequency + Category/Title similarity scoring


## Setup & Load Data

In [ ]:
import numpy as np
import pandas as pd
import zipfile, os
from pathlib import Path
from collections import defaultdict, Counter

# ══════════════════════════════════════════════════════
DATA_PATH = ''  # isi manual jika auto-detect gagal
# ══════════════════════════════════════════════════════

def find_data_root():
    if DATA_PATH:
        p = Path(DATA_PATH)
        if (p / 'articles.csv').exists(): return p
    for z in ['datathon-task-2.zip', 'dataset-task2.zip']:
        if Path(z).exists():
            with zipfile.ZipFile(z) as zf: zf.extractall('.')
            break
    kaggle_in = Path('/kaggle/input')
    if kaggle_in.exists():
        for d in sorted(kaggle_in.iterdir()):
            if not d.is_dir(): continue
            if (d / 'articles.csv').exists(): return d
            for sub in sorted(d.iterdir()):
                if sub.is_dir() and (sub / 'articles.csv').exists(): return sub
    for root in [Path('.'), Path('/content')]:
        if not root.exists(): continue
        if (root / 'articles.csv').exists(): return root
        for d in sorted(root.iterdir()):
            if d.is_dir() and not d.name.startswith('.') and (d / 'articles.csv').exists():
                return d
    return Path('.')

DATA = find_data_root()
OUT  = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

try:
    import google.colab; env = 'Colab'
except ImportError:
    env = 'Kaggle' if Path('/kaggle/input').exists() else 'Local'

print(f'Env: {env} | Data: {DATA.resolve()}')

train = pd.read_csv(DATA / 'states_train.csv')
test  = pd.read_csv(DATA / 'states_test.csv')
arts  = pd.read_csv(DATA / 'articles.csv').set_index('article_id')
cats  = pd.read_csv(DATA / 'categories.csv')

print(f'Train: {train.shape} | Test: {test.shape}')
print(f'Articles: {len(arts)} | Categories: {len(cats)}')
print(train.head(3))


## Feature Engineering

In [ ]:
# Category set per article
art_cats = defaultdict(set)
for _, row in cats.iterrows():
    art_cats[row['article_id']].add(row['category'])

# Title word set per article
art_words = {}
for art_id, row in arts.iterrows():
    art_words[art_id] = set(
        str(row['title']).lower()
        .replace(',', ' ').replace('-', ' ').replace('_', ' ').split()
    )

def cat_sim(a, b):
    ca, cb = art_cats[a], art_cats[b]
    if not ca or not cb: return 0.0
    return len(ca & cb) / len(ca | cb)

def title_sim(a, b):
    wa, wb = art_words.get(a, set()), art_words.get(b, set())
    if not wa or not wb: return 0.0
    return len(wa & wb) / len(wa | wb)

# Transition table: current → Counter(next)
transitions = defaultdict(Counter)
for _, row in train.iterrows():
    transitions[row['current_article_id']][row['next_article_id']] += 1

# Global popularity ranking
global_counter = Counter(train['next_article_id'])
global_top = [a for a, _ in global_counter.most_common()]

print(f'Unique current articles: {len(transitions)}')
print(f'Unique next articles: {len(global_counter)}')


## Prediction Model

In [ ]:
def predict_next(cur, tgt):
    cur, tgt = int(cur), int(tgt)
    
    # Direct: if target is adjacent from current, go there
    if tgt in transitions.get(cur, {}):
        return tgt
    
    if cur in transitions:
        candidates = list(transitions[cur].keys())
        if len(candidates) == 1:
            return candidates[0]
        # Score: category sim + title sim + transition frequency
        freq_sum = max(sum(transitions[cur].values()), 1)
        best_s, pred = -1e9, candidates[0]
        for c in candidates:
            s = (0.5 * cat_sim(c, tgt)
               + 0.3 * title_sim(c, tgt)
               + 0.2 * transitions[cur][c] / freq_sum)
            if s > best_s:
                best_s, pred = s, c
        return pred
    
    # Fallback: global top articles, scored by similarity to target
    top = global_top[:200]
    fs = max(sum(global_counter[c] for c in top), 1)
    best_s, pred = -1e9, top[0]
    for c in top:
        s = (0.5 * cat_sim(c, tgt)
           + 0.3 * title_sim(c, tgt)
           + 0.2 * global_counter[c] / fs)
        if s > best_s:
            best_s, pred = s, c
    return pred

# Internal validation
np.random.seed(42)
val_idx = np.random.choice(len(train), 1500, replace=False)
val = train.iloc[val_idx].reset_index(drop=True)
tr2 = train.drop(val_idx).reset_index(drop=True)

trans2 = defaultdict(Counter)
for _, r in tr2.iterrows():
    trans2[r['current_article_id']][r['next_article_id']] += 1
gc2 = Counter(tr2['next_article_id'])
gt2 = [a for a, _ in gc2.most_common()]

def predict2(cur, tgt):
    cur, tgt = int(cur), int(tgt)
    if tgt in trans2.get(cur, {}):
        return tgt
    if cur in trans2:
        cands = list(trans2[cur].keys())
        if len(cands) == 1: return cands[0]
        fs = max(sum(trans2[cur].values()), 1)
        bs, p = -1e9, cands[0]
        for c in cands:
            s = 0.5*cat_sim(c,tgt) + 0.3*title_sim(c,tgt) + 0.2*trans2[cur][c]/fs
            if s > bs: bs, p = s, c
        return p
    top = gt2[:200]; fs = max(sum(gc2[c] for c in top), 1)
    bs, p = -1e9, top[0]
    for c in top:
        s = 0.5*cat_sim(c,tgt) + 0.3*title_sim(c,tgt) + 0.2*gc2[c]/fs
        if s > bs: bs, p = s, c
    return p

correct = sum(1 for _, r in val.iterrows()
              if predict2(r['current_article_id'], r['target_article_id']) == r['next_article_id'])
print(f'Validation accuracy: {correct/15:.1f}% (on 1500 samples)')


## Generate Submission

In [ ]:
preds = [
    predict_next(r['current_article_id'], r['target_article_id'])
    for _, r in test.iterrows()
]

submission = pd.DataFrame({
    'state_id': test['state_id'],
    'predicted_next_article_id': preds
})

sub_path = OUT / 'submission.csv'
submission.to_csv(sub_path, index=False)

print(f'Submission saved: {sub_path}')
print(f'Rows: {len(submission)}')
print(f'Null: {submission.isnull().sum().sum()}')
print(submission.head())
